# Multi-step (latent-overshooting) training objective — does it change the RSSM's latent world structure?

**Direction `multistep-objective-rssm`** (companion to the GRU result, which was a clean *negative*). The RSSM is trained with a standard ELBO (reconstruction + KL). The question is the **training objective**: does adding **PlaNet-style latent overshooting** — from each posterior state, imagine `W` steps forward through the PRIOR and add reconstruction of the future obs + `KL(sg(posterior) ‖ imagined-prior)` — change the latent's **geometry / recoverability / canonicality / editability** vs the single-step ELBO? The RSSM is the architecture *built for* multi-step latent rollout, so it is the more interesting test of whether the GRU's negative replicates.

Three RSSMs — **identical architecture (det 256 / stoch 64) / data / matched training budget (150 epochs) — only the overshoot horizon `W` changes**:
- **W=1** — standard ELBO (no overshoot).
- **W=2** — latent overshooting to horizon 2.
- **W=5** — latent overshooting to horizon 5.

We replicate the GRU-multistep spread for all three: **§0 sharpness**, **§1 geometry**, **§2 recoverability**, **§3 fiber-collapse** (+ a **det-vs-stoch split** — the deterministic `h` is the primary world-state carrier), **§4 editing head-to-head**. Data `datasets/4_fixed_refl_inview` (T=40, R=128, edit_frame=20, 2 objects), teacher-forced **test** split; edits split for §4.

> **Caveats:** (1) probes are in-sample — the **comparisons across `W`** are the load-bearing quantities. (2) **Reduced 150-epoch budget** (vs the refined RSSM's 500) to fit the compute cap; all three share it so the cross-`W` comparison is clean, but absolute values are undertrained relative to the master's refined RSSM. (3) State = `cat([h_det, s_stoch])` = 320-d; eval runs in **prior-mean mode** (`sample=False`). (4) The tangent-rotation "curvature" is **not distance-normalized** (`directions/curvature-metric-normalization.md`) — its absolute degrees are not comparable across notebooks/architectures; compare only across `W` within this notebook.

In [ ]:
# [1] Bootstrap: imports, config, load THREE RSSMs (w=1 baseline, w=2, w=5), teacher-force, velocities, themes + helpers.
import sys, os
sys.path.insert(0, "../../../..")   # repo root -> import pim

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import h5py

import pim.eval as eval
from pim.eval._helpers import autoregressive_rollouts, autoregressive_rollout
from pim.extractors import LinearExtractor, MLPExtractor, StateDefinition
from pim.editors import (probe_decomposition, inject_state, fit_state_subspace, project_to_subspace,
                         offmanifold_residual, fit_local_subspace, manifold_steer, gradient_steer)
from pim.editors.manifold_steering import _pca_subspace
from pim.eval.controllability import _rollout
from pim.world_models import load_checkpoint, load_dataset, make_test_loader

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE, NUM_WORKERS, N_OBJ = 512, 6, 2
DATA_DIR = "../../../../datasets/4_fixed_refl_inview"
OUT = "/tmp/multistep_objective_rssm"; os.makedirs(OUT, exist_ok=True)

# --- the three checkpoints: ONLY the training objective differs (w = free-run rollout window) ---
CKPTS = {
    "w=1": "../../../../runs/rssm_multistep/w1_dset4/best_model.pt",   # standard ELBO (no overshoot)
    "w=2": "../../../../runs/rssm_multistep/w2_dset4/best_model.pt",   # latent overshoot horizon 2
    "w=5": "../../../../runs/rssm_multistep/w5_dset4/best_model.pt",   # latent overshoot horizon 5
}
WLABELS = list(CKPTS)
WCOLOR = {"w=1": "#0072B2", "w=2": "#D55E00", "w=5": "#009E73"}   # Okabe-Ito: blue / orange / green

# ---- data (shared) ----
bundle = load_dataset(DATA_DIR, n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
test_loader = make_test_loader(test, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
DT = float(test.config["dataset"]["sim"]["dt"])

# ---- load models + teacher-force to get per-timestep hidden states ----
MODELS, INFO, STATES, PREDS = {}, {}, {}, {}
for w, ck in CKPTS.items():
    m, inf = load_checkpoint(ck, device=DEVICE); m.sample = False  # prior-mean deterministic eval
    MODELS[w] = m; INFO[w] = inf
    PREDS[w], STATES[w] = eval.teacher_force(m, test_loader, device=DEVICE)   # (N,39,R), (N,39,256)
H = MODELS["w=1"].hidden_size
DET, STOCH = MODELS["w=1"].cfg.det_size, MODELS["w=1"].cfg.stoch_size
print(f"device={DEVICE}  dt={DT}  H={H} (det {DET} + stoch {STOCH})")
for w in WLABELS:
    _vl = getattr(INFO[w], "val_loss", None)
    print(f"{w}: {INFO[w].run_name} (ep {INFO[w].epoch}, val_recon={_vl if _vl is None else round(_vl,5)})  states={STATES[w].shape}")

# ---- velocities from HDF5 (aligned like positions[:, :-1]); flat (pos,vel) targets; visibility ----
v_test = h5py.File(test.h5_path, "r")["velocities"][:, :, :N_OBJ, :].astype(np.float32)   # (N,40,2,2)
vel_tf = v_test[:, :-1, :, :]                          # (N,39,2,2) aligned with states
pos_tf = test.positions[:, :-1, :N_OBJ, :]            # (N,39,2,2)
vis_tf = test.is_visible[:, :-1, :N_OBJ].all(axis=2)  # (N,39) both objects visible
posflat_tf = pos_tf.reshape(*pos_tf.shape[:2], N_OBJ * 2)          # (N,39,4)
velflat_tf = vel_tf.reshape(*vel_tf.shape[:2], N_OBJ * 2)          # (N,39,4)
posvel_tf = np.concatenate([posflat_tf, velflat_tf], -1)          # (N,39,8)
LATE_T = 15
print("velocity temporal std (constant-vel sim -> ~0):", float(v_test.std(axis=1).mean()))

# ---- themes + shared helpers ----
OK = {"blue":"#0072B2","orange":"#D55E00","green":"#009E73","pink":"#CC79A7","yellow":"#E69F00","grey":"#999999"}
plt.style.use("default")
def style_ax(ax):
    ax.spines[["top","right"]].set_visible(False); ax.grid(alpha=0.25, lw=0.6)

def rms(a, b): return float(np.sqrt(((a - b) ** 2).mean()))
def tv(frames): return float(np.abs(np.diff(frames, axis=-1)).sum(axis=-1).mean())

def _fit_regress(X, Y, kind, hidden=256, n_epochs=100, lr=2e-3, seed=0):
    """Generic probe feats->target: returns (pred, R2_overall, R2_percomp, resid_frac). linear = lstsq, mlp = 2-layer."""
    torch.manual_seed(seed); np.random.seed(seed)
    Din, Dout = X.shape[1], Y.shape[1]
    Xt = torch.from_numpy(X.astype(np.float32)).to(DEVICE); Yt = torch.from_numpy(Y.astype(np.float32)).to(DEVICE)
    if kind == "linear":
        Xa = torch.cat([Xt, torch.ones(Xt.shape[0], 1, device=DEVICE)], 1)
        sol = torch.linalg.lstsq(Xa, Yt).solution
        with torch.no_grad(): pred = Xa @ sol
    else:
        net = nn.Sequential(nn.Linear(Din, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(),
                            nn.Linear(hidden, Dout)).to(DEVICE)
        opt = torch.optim.Adam(net.parameters(), lr=lr); bs = 4096; Nn = Xt.shape[0]
        for ep in range(n_epochs):
            perm = torch.randperm(Nn, device=DEVICE)
            for i in range(0, Nn, bs):
                idx = perm[i:i+bs]; loss = ((net(Xt[idx]) - Yt[idx]) ** 2).mean()
                opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): pred = net(Xt)
    resid2 = ((pred - Yt) ** 2).sum(0); tot2 = ((Yt - Yt.mean(0, keepdim=True)) ** 2).sum(0)
    r2_pc = (1 - resid2 / torch.clamp(tot2, min=1e-12)).cpu().numpy()
    r2_all = float(1 - resid2.sum() / tot2.sum())
    resid_frac = float(((((pred - Yt) ** 2).sum() / (Yt ** 2).sum())).sqrt())
    return pred.cpu().numpy(), r2_all, r2_pc, resid_frac

def fit_probe(feats_tf, y_tf, mask, kind, **kw):
    """feats_tf:(N,T,F) y_tf:(N,T,D) mask:(N,T) bool. Fit on masked entries, return dict."""
    X = feats_tf[mask]; Y = y_tf[mask]
    pred, r2, r2pc, rfrac = _fit_regress(X, Y, kind, **kw)
    return dict(r2=r2, r2pc=r2pc, resid_frac=rfrac, n=X.shape[0])

def md_table(data, columns, row_hdr=""):
    """Rendered-markdown table. data: {row: {key: value}}; columns: list of (key, display, fmt)."""
    lines = ["| " + row_hdr + " | " + " | ".join(d for _, d, _ in columns) + " |",
             "|" + "---|" * (len(columns) + 1)]
    for rn, vals in data.items():
        cells = []
        for k, _, f in columns:
            v = vals[k]
            cells.append("nan" if isinstance(v, float) and np.isnan(v) else f.format(v))
        lines.append("| **" + str(rn) + "** | " + " | ".join(cells) + " |")
    return Markdown("\n".join(lines))

print("bootstrap ready: 3 RSSMs loaded, teacher-forced, velocities + themes + probe helpers defined.")


---
## Definitions & metrics (read once)

Every non-obvious term with its formula, units, and better-direction. The three objectives are the only thing that varies; every metric is computed identically across `w`.

| symbol / metric | definition & formula | units | better |
|---|---|---|---|
| `w` | free-run rollout window in the **training** objective (`w=1` = single-step baseline; `w=2`, `w=5` = multi-step). All else identical | steps | — |
| `(pos, vel)` | per-object position & velocity; the physical minimal sufficient statistic = 2 obj × (2 pos + 2 vel) = **8-dim** | — | — |
| `h` | RSSM hidden state under test, `h ∈ ℝ³²⁰ (det 256 + stoch 64)` | — | — |
| next-step RMSE | `√mean((pred − target)²)` for teacher-forced 1-step prediction; target = **clean** (noiseless) obs unless noted | obs intensity | ↓ |
| open-loop horizon RMSE | free-run rollout (warm up `NC=10` real frames, then feed own predictions); per-step RMSE vs clean obs | obs intensity | ↓ |
| rollout TV sharpness | per-frame spatial total variation `mean Σ|Δ_ray obs|` of the free-run region; blurry mean-hedging ⇒ **lower** TV than the sim's GT | intensity/frame | → GT |
| PCA hull dim @p% | smallest #PCA comps of the visited-`h` bank with cumulative variance ≥ p%; linear-hull upper bound | dims | — |
| intrinsic dim (TwoNN / MLE) | model-free estimators (Facco 2017 / Levina–Bickel), no linearity assumed | dims | — |
| tangent rotation | mean principal angle between local-PCA tangents (k=64 NN, top-8) of an anchor and its nearest neighbour; 0° = flat | deg | ↓ flatter |
| position / velocity R² | `1 − SS_resid/SS_tot` of a probe `h → (pos or vel)`; linear (lstsq) or MLP | — | ↑ |
| fiber residual | `‖h − g(pos,vel)‖ / ‖h‖`, `g` linear/MLP; fraction of `h` **not** a function of the 8-dim statistic (0 = fully canonical) | frac of ‖h‖ | ↓ |
| §4 editing metrics | defined in the §4 metric table below (readout RMSE, GT next-step RMSE, obs-change %-of-swap, ghost-ray ratio, leave-out local-PCA residual) | — | — |

*Colour key (all figures): `w=1` blue, `w=2` orange, `w=5` green.*

In [ ]:
# [2] §0 — Sharpness & next-step predictive quality (the blur / mode-collapse WATCH-ITEM).
#   (i) teacher-forced next-step RMSE vs CLEAN obs (does multi-step help/hurt the 1-step metric?);
#   (ii) free-run OPEN-LOOP horizon RMSE vs clean; (iii) total-variation sharpness of the free-run rollout region
#        (blurry mean-hedging => low TV vs the sim's GT). Reported per objective.
clean = test.clean_obs; obs_np = test.obs
NEXT = {}
for w in WLABELS:
    NEXT[w] = dict(rmse_clean=rms(PREDS[w], clean[:, 1:, :]), rmse_noisy=rms(PREDS[w], obs_np[:, 1:, :]), tv=tv(PREDS[w]))
GT_TV_next = tv(clean[:, 1:, :])

NC, KR = 10, 800
tgt_roll = clean[:KR, NC:, :]
HRMSE, TV_ROLL = {}, {}
roll = {}
for w in WLABELS:
    roll[w] = autoregressive_rollouts(MODELS[w], obs_np[:KR], n_context=NC, device=DEVICE, desc=f"open-loop {w}")
    HRMSE[w] = np.sqrt(((roll[w] - tgt_roll) ** 2).mean(axis=(0, 2)))   # per-step RMSE
    TV_ROLL[w] = tv(roll[w])
GT_TV_ROLL = tv(tgt_roll)
persist = np.repeat(obs_np[:KR, NC-1:NC, :], tgt_roll.shape[1], axis=1)
persist_rmse = np.sqrt(((persist - tgt_roll) ** 2).mean(axis=(0, 2)))

# ---- table ----
print("§0 — sharpness & next-step predictive quality per objective (RMSE vs clean; TV = per-frame spatial total variation).")
S_ROWS = {w: dict(next_clean=NEXT[w]["rmse_clean"], next_noisy=NEXT[w]["rmse_noisy"],
                  horizon=float(np.mean(HRMSE[w])), tv_roll=TV_ROLL[w], tv_ratio=TV_ROLL[w]/GT_TV_ROLL) for w in WLABELS}
S_TCOLS = [("next_clean","next-step RMSE vs clean","{:.4f}"), ("next_noisy","next-step RMSE vs noisy","{:.4f}"),
           ("horizon","open-loop horizon RMSE (mean)","{:.4f}"), ("tv_roll","rollout TV (sharpness)","{:.3f}"),
           ("tv_ratio","rollout TV / GT TV","{:.3f}")]
display(md_table(S_ROWS, S_TCOLS, row_hdr="objective"))
print(f"GT rollout TV (sim clean) = {GT_TV_ROLL:.3f}; GT next-step TV = {GT_TV_next:.3f}. TV ratio near 1 = as sharp as GT; <1 = blurrier.")
for w in WLABELS:
    print(f"  {w}: next-step RMSE(clean)={NEXT[w]['rmse_clean']:.4f}  horizon RMSE={np.mean(HRMSE[w]):.4f}  TV/GT={TV_ROLL[w]/GT_TV_ROLL:.3f}")

# ---- Fig 0 ----
fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.4))
ax = axes[0]
xs = np.arange(1, tgt_roll.shape[1] + 1)
for w in WLABELS:
    ax.plot(xs, HRMSE[w], color=WCOLOR[w], lw=2, marker="o", ms=3, label=w)
ax.plot(xs, persist_rmse, color="0.5", ls=":", lw=1.5, label="persistence")
ax.set_xlabel("open-loop rollout step"); ax.set_ylabel("clean-obs RMSE"); ax.set_title("(a) open-loop horizon RMSE (lower = better)")
ax.legend(fontsize=8); style_ax(ax)
ax = axes[1]
xw = np.arange(len(WLABELS))
ax.bar(xw, [NEXT[w]["rmse_clean"] for w in WLABELS], color=[WCOLOR[w] for w in WLABELS], alpha=0.9)
for xi, w in zip(xw, WLABELS): ax.text(xi, NEXT[w]["rmse_clean"] + 0.0005, f"{NEXT[w]['rmse_clean']:.4f}", ha="center", fontsize=8)
ax.set_xticks(xw); ax.set_xticklabels(WLABELS); ax.set_ylabel("teacher-forced next-step RMSE (vs clean)")
ax.set_title("(b) next-step predictive quality"); style_ax(ax)
ax = axes[2]
ax.bar(xw, [TV_ROLL[w] for w in WLABELS], color=[WCOLOR[w] for w in WLABELS], alpha=0.9)
for xi, w in zip(xw, WLABELS): ax.text(xi, TV_ROLL[w] + 0.02, f"{TV_ROLL[w]:.2f}", ha="center", fontsize=8)
ax.axhline(GT_TV_ROLL, color="k", ls="--", lw=1.4, label=f"GT (sim) TV {GT_TV_ROLL:.2f}")
ax.set_xticks(xw); ax.set_xticklabels(WLABELS); ax.set_ylabel("rollout total-variation sharpness")
ax.set_title("(c) rollout sharpness vs GT (blur check)"); ax.legend(fontsize=8); style_ax(ax)
fig.suptitle("Fig 0 — Sharpness & next-step predictive quality: does the multi-step objective blur the decoder?", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig0_sharpness.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)


---
## §1 — Geometry: how many DOF does the visited-state manifold have, and is it flat or curved?

Three reads on the bank of visited hidden states `h`, per objective: (1) **linear-hull dimension** (# PCA components for 90% variance — an upper bound); (2) **intrinsic dimension** (model-free **TwoNN** and **MLE**, no linearity assumed); (3) **curvature** (mean principal angle a local tangent rotates between neighbouring states — recomputed here on each model's own bank, not cited). All read against the physical **8 DOF**. Curvature is the geometric reason min-norm linear edits leave the manifold (→ §4), so a straighter manifold under the multi-step objective would be directly relevant to editability.

In [ ]:
# [3] §1 — Geometry: PCA scree, model-free intrinsic dim (TwoNN + MLE), and tangent-rotation curvature, per objective.
PHYS_DOF = 8

def scree(states):
    bank = states.reshape(-1, H)
    sub = _pca_subspace(torch.from_numpy(bank).float().to(DEVICE), n_components=H, var_threshold=1.0)
    ratio = sub.explained_variance_ratio.cpu().numpy(); cum = np.cumsum(ratio)
    return cum, {p: int((cum < p).sum()) + 1 for p in (0.70, 0.90, 0.95)}

@torch.no_grad()
def _knn_dists(Q, X, k, chunk=2000):
    out = []
    for i in range(0, Q.shape[0], chunk):
        d = torch.cdist(Q[i:i+chunk], X); vals, _ = torch.topk(d, k + 1, largest=False, dim=1); out.append(vals)
    return torch.cat(out, 0)

@torch.no_grad()
def two_nn_id(X, sample=20000, seed=0):
    g = torch.Generator(device=X.device).manual_seed(seed)
    idx = torch.randperm(X.shape[0], generator=g, device=X.device)[:min(sample, X.shape[0])]
    vals = _knn_dists(X[idx], X, 2); r1, r2 = vals[:, 1], vals[:, 2]
    keep = (r1 > 1e-9) & (r2 > r1); return float(1.0 / torch.log(r2[keep] / r1[keep]).mean())

@torch.no_grad()
def mle_id(X, k=20, sample=20000, seed=0):
    g = torch.Generator(device=X.device).manual_seed(seed)
    idx = torch.randperm(X.shape[0], generator=g, device=X.device)[:min(sample, X.shape[0])]
    vals = _knn_dists(X[idx], X, k); Tk = vals[:, 1:k+1].clamp_min(1e-9); logT = torch.log(Tk)
    m_inv = (logT[:, k-1:k] - logT[:, :k-1]).mean(1); mk = 1.0 / m_inv.clamp_min(1e-9)
    return float(mk.mean() * (k - 2) / (k - 1))

def id_bank(states, cap=200_000, seed=0):
    bank = states.reshape(-1, H); rng = np.random.RandomState(seed)
    idx = rng.choice(bank.shape[0], size=min(cap, bank.shape[0]), replace=False)
    return torch.from_numpy(bank[idx]).float().to(DEVICE)

@torch.no_grad()
def tangent_rotation(bank, n_anchor=60, k=64, n_comp=8, seed=0):
    """Mean principal angle between local-PCA tangents (top-n_comp) of an anchor and its nearest neighbour. 0deg=flat."""
    g = torch.Generator(device=bank.device).manual_seed(seed)
    aidx = torch.randperm(bank.shape[0], generator=g, device=bank.device)[:n_anchor]
    angs = []
    for a in aidx:
        d = torch.cdist(bank[a][None], bank)[0]; nn = torch.topk(d, k + 1, largest=False).indices
        subA = _pca_subspace(bank[nn], n_components=n_comp, var_threshold=1.0)
        d2 = torch.cdist(bank[nn[1]][None], bank)[0]; nn2 = torch.topk(d2, k + 1, largest=False).indices
        subB = _pca_subspace(bank[nn2], n_components=n_comp, var_threshold=1.0)
        s = torch.linalg.svdvals(subA.basis.T @ subB.basis).clamp(-1, 1)
        angs.append(float(torch.rad2deg(torch.arccos(s)).mean()))
    return float(np.mean(angs))

GEO = {}
for w in WLABELS:
    cum, dims = scree(STATES[w]); bank = id_bank(STATES[w])
    GEO[w] = dict(cum=cum, dims=dims, twonn=two_nn_id(bank), mle=mle_id(bank, k=20), tangent=tangent_rotation(bank))
    del bank
    if DEVICE == "cuda": torch.cuda.empty_cache()

print("§1 GEOMETRY — dimensionality & curvature of the visited-state bank, per objective (physical DOF = 8).")
G_ROWS = {w: dict(hull70=GEO[w]["dims"][0.70], hull90=GEO[w]["dims"][0.90], hull95=GEO[w]["dims"][0.95],
                  twonn=GEO[w]["twonn"], mle=GEO[w]["mle"], tangent=GEO[w]["tangent"]) for w in WLABELS}
G_TCOLS = [("hull70","PCA hull @70%","{:d}"), ("hull90","PCA hull @90%","{:d}"), ("hull95","PCA hull @95%","{:d}"),
           ("twonn","intrinsic dim TwoNN","{:.2f}"), ("mle","intrinsic dim MLE","{:.2f}"), ("tangent","tangent rotation @NN (deg)","{:.1f}")]
display(md_table(G_ROWS, G_TCOLS, row_hdr="objective"))
print(f"{'objective':10s} {'hull90':>7s} {'TwoNN':>7s} {'MLE':>7s} {'tangent°':>9s}")
for w in WLABELS:
    print(f"{w:10s} {GEO[w]['dims'][0.90]:7d} {GEO[w]['twonn']:7.2f} {GEO[w]['mle']:7.2f} {GEO[w]['tangent']:9.1f}")
print("Read: compare intrinsic dim / curvature across w — a lower intrinsic dim or flatter tangent under the "
      "multi-step objective would mean a simpler / straighter learned manifold.")


In [ ]:
# [4] Fig 1 — §1 geometry across objectives: (a) PCA scree, (b) intrinsic dim (TwoNN, MLE) vs linear hull@90%, (c) curvature.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

ax = axes[0]
for w in WLABELS:
    cum = GEO[w]["cum"]
    ax.plot(np.arange(1, len(cum) + 1), cum, color=WCOLOR[w], lw=2, label=w)
    ax.axvline(GEO[w]["dims"][0.90], color=WCOLOR[w], ls="--", lw=1)
ax.axhline(0.90, color="0.6", ls=":", lw=1); ax.set_xlim(0, 80)
ax.set_xlabel("# PCA components"); ax.set_ylabel("cumulative variance")
ax.set_title("(a) PCA scree (dashed = hull@90% per objective)"); ax.legend(fontsize=8); style_ax(ax)

ax = axes[1]
cats = ["TwoNN", "MLE", "hull@90%"]; x = np.arange(len(cats)); nM = len(WLABELS); w_ = 0.8 / nM
for j, w in enumerate(WLABELS):
    off = (j - (nM - 1) / 2) * w_
    vals = [GEO[w]["twonn"], GEO[w]["mle"], GEO[w]["dims"][0.90]]
    ax.bar(x + off, vals, width=w_, color=WCOLOR[w], alpha=0.9, label=w)
    for xi, v in zip(x + off, vals): ax.text(xi, v + 0.6, f"{v:.1f}", ha="center", fontsize=7.5)
ax.axhline(PHYS_DOF, color="k", ls="--", lw=1.5, label=f"physical {PHYS_DOF} DOF")
ax.set_xticks(x); ax.set_xticklabels(cats); ax.set_ylabel("dimension")
ax.set_title("(b) intrinsic dim (TwoNN, MLE) vs linear hull@90%"); ax.legend(fontsize=8); style_ax(ax)

ax = axes[2]
tvals = [GEO[w]["tangent"] for w in WLABELS]
ax.bar(WLABELS, tvals, color=[WCOLOR[w] for w in WLABELS], alpha=0.9)
for xi, v in enumerate(tvals): ax.text(xi, v + 1, f"{v:.0f}°", ha="center", fontsize=8)
ax.set_ylim(0, max(tvals) * 1.25 + 5)
ax.set_ylabel("tangent rotation @ NN spacing (deg)")
ax.set_title("(c) local tangent reorientation (curvature; recomputed)"); style_ax(ax)

fig.suptitle("Fig 1 — State geometry across training objectives: intrinsic dim, linear hull, curvature", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_geometry.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)


---
## §2 — Recoverability: can `(pos, vel)` be read out of a single hidden state `h`?

Given one hidden state `h_t`, can a probe recover the physical statistic? A **linear** least-squares map vs a 2-layer **MLP**, on **position** (`h_t → pos`) and **velocity** (`→ vel`). Velocity is a 2×2 of {linear, MLP} × {single-frame `h_t`, two-frame `[h_{t-1}, h_t]`} plus a differenced control `dh = h_t − h_{t-1}`. Because the world is noisy (the model runs an implicit filter), we split **early-t (t<15, belief not converged)** vs **late-t (t≥15, converged)**. The multi-step question: does forcing coherence under iterated dynamics make `(pos,vel)` more (or less) linearly readable from a single state?

In [ ]:
# [5] §2 — recoverability of (pos,vel) from a single hidden state h: linear vs MLP probes, single- vs two-frame,
#     early-t (t<15) vs late-t (t≥15), per objective. Velocity 2x2 + differenced-dh control; position linear/MLP.
def build_feats(states, regime):
    sf = states[:, 1:, :]                                              # single-frame h_t
    win = np.concatenate([states[:, :-1, :], states[:, 1:, :]], -1)    # two-frame [h_{t-1}, h_t]
    dh = states[:, 1:, :] - states[:, :-1, :]                          # differenced control
    y = velflat_tf[:, 1:, :]
    mask = vis_tf[:, 1:] & vis_tf[:, :-1]
    rm = np.zeros_like(mask)
    if regime == "early": rm[:, :LATE_T - 1] = True
    else: rm[:, LATE_T - 1:] = True
    return sf, win, dh, y, (mask & rm)

def run_vel_2x2(states, regime):
    sf, win, dh, y, mask = build_feats(states, regime)
    return {("sf","lin"): fit_probe(sf, y, mask, "linear"), ("sf","mlp"): fit_probe(sf, y, mask, "mlp"),
            ("win","lin"): fit_probe(win, y, mask, "linear"), ("win","mlp"): fit_probe(win, y, mask, "mlp"),
            ("dh","mlp"): fit_probe(dh, y, mask, "mlp")}

VEL, POS = {}, {}
for w in WLABELS:
    for reg in ["early", "late"]:
        VEL[(w, reg)] = run_vel_2x2(STATES[w], reg)
    POS[(w, "lin")] = fit_probe(STATES[w], posflat_tf, vis_tf, "linear")
    POS[(w, "mlp")] = fit_probe(STATES[w], posflat_tf, vis_tf, "mlp")

reg_label = {"early": "early-t (t<15)", "late": "late-t (t≥15)"}
VCOLS = ["single-frame linear", "single-frame MLP", "two-frame linear", "two-frame MLP", "differenced dh (MLP)", "two-frame − single-frame (MLP)"]
def _vrow(w, reg):
    o = VEL[(w, reg)]
    return [o[("sf","lin")]["r2"], o[("sf","mlp")]["r2"], o[("win","lin")]["r2"], o[("win","mlp")]["r2"],
            o[("dh","mlp")]["r2"], o[("win","mlp")]["r2"] - o[("sf","mlp")]["r2"]]
print("Velocity readout R² (overall) per objective — single-frame vs two-frame = instantaneous vs temporal; linear vs MLP = nonlinearity.")
_md = "| objective | frame regime | " + " | ".join(VCOLS) + " |\n|---|---|" + "|".join(["---"]*len(VCOLS)) + "|\n"
for w in WLABELS:
    for reg in ["early", "late"]:
        _md += f"| {w} | {reg_label[reg]} | " + " | ".join(f"{v:.3f}" for v in _vrow(w, reg)) + " |\n"
display(Markdown(_md))
print(f"{'objective / regime':22s} {'sf-lin':>7s} {'sf-MLP':>7s} {'2f-lin':>7s} {'2f-MLP':>7s} {'dh-MLP':>7s} {'2f-sf':>7s}")
for w in WLABELS:
    for reg in ["early", "late"]:
        print(f"{w+' '+reg_label[reg]:22s} " + " ".join(f"{v:7.3f}" for v in _vrow(w, reg)))

print("\nPosition readout R² (overall) per objective — how well can position be read from a single h? Higher = better.")
_md2 = "| objective | linear probe R² | MLP probe R² |\n|---|---|---|\n"
for w in WLABELS:
    _md2 += f"| {w} | {POS[(w,'lin')]['r2']:.3f} | {POS[(w,'mlp')]['r2']:.3f} |\n"
display(Markdown(_md2))
print("Read: compare probe R² across w — a change would mean the multi-step objective altered how readable (pos,vel) is from one state.")


In [ ]:
# [6] Fig 2 — §2 recoverability of (pos,vel) from a single hidden state h, across objectives.
#   (a) velocity single-frame linear vs MLP (late-t); (b) position linear vs MLP; (c) velocity early-t vs late-t (MLP).
fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
xw = np.arange(len(WLABELS)); bw = 0.38

ax = axes[0]
vlin = [VEL[(w,"late")][("sf","lin")]["r2"] for w in WLABELS]
vmlp = [VEL[(w,"late")][("sf","mlp")]["r2"] for w in WLABELS]
ax.bar(xw - bw/2, vlin, bw, color=OK["blue"], label="linear probe")
ax.bar(xw + bw/2, vmlp, bw, color=OK["orange"], label="MLP probe")
for xi, v in zip(xw - bw/2, vlin): ax.text(xi, v + 0.015, f"{v:.2f}", ha="center", fontsize=7.5)
for xi, v in zip(xw + bw/2, vmlp): ax.text(xi, v + 0.015, f"{v:.2f}", ha="center", fontsize=7.5)
ax.set_xticks(xw); ax.set_xticklabels(WLABELS); ax.set_ylim(0, 1.1)
ax.set_ylabel("velocity R² (single-frame, late-t)"); ax.set_title("(a) velocity from a single frame h_t")
ax.legend(fontsize=8); style_ax(ax)

ax = axes[1]
plin = [POS[(w,"lin")]["r2"] for w in WLABELS]; pmlp = [POS[(w,"mlp")]["r2"] for w in WLABELS]
ax.bar(xw - bw/2, plin, bw, color=OK["blue"], label="linear probe")
ax.bar(xw + bw/2, pmlp, bw, color=OK["orange"], label="MLP probe")
for xi, v in zip(xw - bw/2, plin): ax.text(xi, v + 0.015, f"{v:.2f}", ha="center", fontsize=7.5)
for xi, v in zip(xw + bw/2, pmlp): ax.text(xi, v + 0.015, f"{v:.2f}", ha="center", fontsize=7.5)
ax.set_xticks(xw); ax.set_xticklabels(WLABELS); ax.set_ylim(0, 1.1)
ax.set_ylabel("position R²"); ax.set_title("(b) position: linear vs MLP")
ax.legend(fontsize=8); style_ax(ax)

ax = axes[2]
vearly = [VEL[(w,"early")][("sf","mlp")]["r2"] for w in WLABELS]
vlate = [VEL[(w,"late")][("sf","mlp")]["r2"] for w in WLABELS]
ax.bar(xw - bw/2, vearly, bw, color="#999999", label="early-t (t<15)")
ax.bar(xw + bw/2, vlate, bw, color=OK["green"], label="late-t (t≥15)")
for xi, v in zip(xw - bw/2, vearly): ax.text(xi, v + 0.015, f"{v:.2f}", ha="center", fontsize=7.5)
for xi, v in zip(xw + bw/2, vlate): ax.text(xi, v + 0.015, f"{v:.2f}", ha="center", fontsize=7.5)
ax.set_xticks(xw); ax.set_xticklabels(WLABELS); ax.set_ylim(0, 1.1)
ax.set_ylabel("velocity R² (single-frame MLP)"); ax.set_title("(c) velocity readout: filter convergence (early vs late-t)")
ax.legend(fontsize=8); style_ax(ax)

fig.suptitle("Fig 2 — Recoverability of (pos, vel) from a single hidden state h, across training objectives", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_recoverability.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)


---
## §3 — Canonicality / fiber-collapse: is the hidden state a *function* of `(pos, vel)`?

A hidden state is **canonical** w.r.t. the physical statistic if it is a *function* of `(pos, vel)` — every state with the same `(pos, vel)` maps to the same `h`. We fit the best map `g(pos,vel) → h` (linear and MLP) and measure the **fiber residual** `‖h − g(pos,vel)‖ / ‖h‖`: the fraction of `h` that **no** function of the 8-dim statistic can explain (0 = fully canonical). The multi-step hypothesis: forcing `h` to be coherent under its own iterated dynamics might collapse the off-`(pos,vel)` fiber (lower residual). *(Caveat from §0 of the master notebook: this is measured against the 8-dim* physical *statistic, not the belief/causal state, which is legitimately >8-dim under noise.)*

In [ ]:
# [7] §3 — fiber-collapse residual: fit g(pos,vel)->h (linear & MLP), residual fraction ‖h − g‖/‖h‖ per objective.
def fit_g(states):
    X = posvel_tf[vis_tf]
    Y = states[vis_tf]
    out = {}
    for kind in ["linear", "mlp"]:
        _, r2, _, rfrac = _fit_regress(X, Y, kind, hidden=512, n_epochs=120, lr=1.5e-3)
        out[kind] = (rfrac, r2)
    return out

FIBER = {w: fit_g(STATES[w]) for w in WLABELS}

print("Fiber-collapse residual — is h a function of the 8-dim (pos,vel)? residual = ‖h − g(pos,vel)‖ / ‖h‖ "
      "(lower = more canonical). A large linear→MLP drop signals a curved embedding.")
FIB_ROWS = {w: dict(lin_rf=FIBER[w]["linear"][0], lin_r2=FIBER[w]["linear"][1],
                    mlp_rf=FIBER[w]["mlp"][0], mlp_r2=FIBER[w]["mlp"][1]) for w in WLABELS}
FIB_TCOLS = [("lin_rf","linear g: residual frac","{:.3f}"), ("lin_r2","linear g: R² on h","{:.3f}"),
             ("mlp_rf","MLP g: residual frac","{:.3f}"), ("mlp_r2","MLP g: R² on h","{:.3f}")]
display(md_table(FIB_ROWS, FIB_TCOLS, row_hdr="objective"))
print(f"{'objective':12s} {'lin resid':>10s} {'lin R²':>8s} {'MLP resid':>10s} {'MLP R²':>8s}")
for w in WLABELS:
    print(f"{w:12s} {FIBER[w]['linear'][0]:10.3f} {FIBER[w]['linear'][1]:8.3f} {FIBER[w]['mlp'][0]:10.3f} {FIBER[w]['mlp'][1]:8.3f}")
print("Read: compare MLP residual fraction across w — a lower value under the multi-step objective would mean h "
      "is MORE nearly a function of the physical state (more canonical).")


In [ ]:
# [8] Fig 3 — §3 fiber-collapse: is h a function of the 8-dim (pos,vel)?  (a) residual fraction, (b) R² of g on h.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
x = np.arange(len(WLABELS)); wbar = 0.38
ax = axes[0]
lin_rf = [FIBER[w]["linear"][0] for w in WLABELS]; mlp_rf = [FIBER[w]["mlp"][0] for w in WLABELS]
ax.bar(x - wbar/2, lin_rf, wbar, label="linear g", color=OK["blue"])
ax.bar(x + wbar/2, mlp_rf, wbar, label="MLP g", color=OK["orange"])
for xi, v in zip(x - wbar/2, lin_rf): ax.text(xi, v + 0.008, f"{v:.3f}", ha="center", fontsize=8, color=OK["blue"])
for xi, v in zip(x + wbar/2, mlp_rf): ax.text(xi, v + 0.008, f"{v:.3f}", ha="center", fontsize=8, color=OK["orange"])
ax.axhline(FIBER["w=1"]["mlp"][0], color=OK["green"], ls=":", lw=1.2, label=f"w=1 MLP reference ({FIBER['w=1']['mlp'][0]:.3f})")
ax.set_xticks(x); ax.set_xticklabels(WLABELS); ax.set_ylim(0, max(lin_rf) * 1.25)
ax.set_ylabel("residual fraction ‖h − g‖ / ‖h‖"); ax.set_title("(a) is h a function of (pos,vel)? (lower = more canonical)")
ax.legend(fontsize=8); style_ax(ax)
ax = axes[1]
lin_r2 = [FIBER[w]["linear"][1] for w in WLABELS]; mlp_r2 = [FIBER[w]["mlp"][1] for w in WLABELS]
ax.bar(x - wbar/2, lin_r2, wbar, label="linear g", color=OK["blue"])
ax.bar(x + wbar/2, mlp_r2, wbar, label="MLP g", color=OK["orange"])
for xi, v in zip(x - wbar/2, lin_r2): ax.text(xi, v + 0.008, f"{v:.3f}", ha="center", fontsize=8, color=OK["blue"])
for xi, v in zip(x + wbar/2, mlp_r2): ax.text(xi, v + 0.008, f"{v:.3f}", ha="center", fontsize=8, color=OK["orange"])
ax.set_xticks(x); ax.set_xticklabels(WLABELS); ax.set_ylim(0, 1.0)
ax.set_ylabel("R² of g(pos,vel) on h"); ax.set_title("(b) variance of h explained by g(pos,vel)")
ax.legend(fontsize=8); style_ax(ax)
fig.suptitle("Fig 3 — Canonicality / fiber collapse across training objectives", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_fiber.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)


---
## §3b — Deterministic `h` vs stochastic `s`: which part carries the world state, and does the objective move it?

The RSSM state is `cat([h_det (256), s_stoch (64)])`. The deterministic `h` is expected to carry essentially all the physical `(pos,vel)` code and be the more canonical part (the GRU-comparison work found det-only position readout on par with the full state). We report, per objective `W` and per slice {full / det / stoch}, the **linear** readability of position & velocity, the **MLP fiber residual** (canonicality), and the **TwoNN** intrinsic dim.

In [ ]:
# [8b] §3b — det vs stoch split: (pos,vel) LINEAR readability + MLP fiber residual + TwoNN dim, per objective & slice.
SLICES = {"full (320)": (0, H), "det h (256)": (0, DET), "stoch s (64)": (DET, H)}
_vmask = (vis_tf[:, 1:] & vis_tf[:, :-1]).copy(); _rm = np.zeros_like(_vmask); _rm[:, LATE_T - 1:] = True; _vmask &= _rm
_yvel = velflat_tf[:, 1:, :]

DS = {}
for w in WLABELS:
    for part, (lo, hi) in SLICES.items():
        S = STATES[w][..., lo:hi]
        pos_lin = fit_probe(S, posflat_tf, vis_tf, "linear")["r2"]
        vel_lin = fit_probe(S[:, 1:, :], _yvel, _vmask, "linear")["r2"]
        _, _, _, fib = _fit_regress(posvel_tf[vis_tf], S[vis_tf], "mlp", hidden=512, n_epochs=80, lr=1.5e-3)
        d = hi - lo; bk = S.reshape(-1, d)
        idx = np.random.RandomState(0).choice(bk.shape[0], size=min(120000, bk.shape[0]), replace=False)
        tw = two_nn_id(torch.from_numpy(bk[idx]).float().to(DEVICE))
        DS[(w, part)] = dict(pos_lin=pos_lin, vel_lin=vel_lin, fiber=fib, twonn=tw)
    if DEVICE == "cuda": torch.cuda.empty_cache()

print("§3b det vs stoch — (pos,vel) linear readability + canonicality + intrinsic dim, per objective and state slice.")
_md = "| objective | state slice | pos R² (linear) | vel R² (linear, late) | MLP fiber residual | TwoNN dim |\n|---|---|---|---|---|---|\n"
for w in WLABELS:
    for part in SLICES:
        o = DS[(w, part)]
        _md += f"| {w} | {part} | {o['pos_lin']:.3f} | {o['vel_lin']:.3f} | {o['fiber']:.3f} | {o['twonn']:.2f} |\n"
display(Markdown(_md))
print("Read: det h should carry ~all the (pos,vel) code and be more canonical than stoch s; compare across W within a slice.")

fig, axes = plt.subplots(1, 3, figsize=(17, 4.4)); xw = np.arange(len(WLABELS)); parts = list(SLICES); nbp = len(parts); bw = 0.8 / nbp
pcolor = {"full (320)": OK["grey"], "det h (256)": OK["blue"], "stoch s (64)": OK["orange"]}
for ax, key, ylab, ttl in [(axes[0], "pos_lin", "position R² (linear)", "(a) position linear-readability"),
                           (axes[1], "vel_lin", "velocity R² (linear, late)", "(b) velocity linear-readability"),
                           (axes[2], "fiber", "MLP fiber residual", "(c) canonicality (lower = more canonical)")]:
    for j, part in enumerate(parts):
        off = (j - (nbp - 1) / 2) * bw
        ax.bar(xw + off, [DS[(w, part)][key] for w in WLABELS], width=bw, color=pcolor[part], label=part)
    ax.set_xticks(xw); ax.set_xticklabels(WLABELS); ax.set_ylabel(ylab); ax.set_title(ttl); ax.legend(fontsize=7); style_ax(ax)
fig.suptitle("Fig 3b — Deterministic h vs stochastic s: (pos,vel) readability + canonicality across objectives", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3b_det_stoch.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)

---
## §4 — Editing head-to-head (the headline): does the multi-step objective make the latent more editable?

All five editors run on the **same** edit set for all three objectives. Teacher-force each edits-split sequence to `edit_frame=20` on the **pre-edit** observations, apply the editor to the hidden state `h`, then roll the model out freely for 15 steps. The intended outcome is the simulation's true post-edit trajectory (`edits.clean_obs`, teleport applied) — the **GT (sim)** reference, never a model output.

**References (not editors):** **GT (sim)** = the simulation's clean post-edit observations; **Unsteered** = rollout from the un-edited warm-up state `h0`; **True-state swap** = rollout from the teacher-forced post-edit state (the model has *seen* the teleport frame) — the upper bound any hidden-state editor could reach.

**Editors:** **Readout injection** (linear-probe pseudoinverse), **MLP-probe gradient** (Adam on `h` through a frozen MLP (pos,vel) probe), **Global-PCA projection** (POCS: inject ↔ project onto global 90%-var PCA subspace), **PCA geodesic** (iterative constant-step walk re-projecting onto a fresh local-PCA tangent each step, K=120), **Decoder gradient** (ORACLE: Adam on `h` to match the GT observation of the edit frame through the decoder).

### §4 metric definitions

| metric | formula | units | better |
|---|---|---|---|
| readout RMSE | `√mean((A·h_edit + b − target_pos)²)` at the edit step, before any rollout | position | ↓ |
| GT next-step RMSE | `RMSE(gen obs @ rollout step 1, sim clean obs @ frame ef+1)` | obs intensity | ↓ |
| per-step GT-trajectory RMSE | `RMSE(gen obs @ step s, sim clean obs @ frame ef+s)` | obs intensity | ↓ |
| step-0 →static-target RMSE | `RMSE(gen obs @ step 0, static render of the edit-frame target)` — step-0 direct-edit check only | obs intensity | ↓ |
| obs-change (% of swap) | `100 · RMSE(obs_edit, obs_unsteered) / RMSE(obs_swap, obs_unsteered)` at step 0 | % | → 100 |
| ghost-ray ratio | mean step-0 intensity on ghost rays (edited object pre-edit but not post-edit), editor ÷ unsteered | ratio | ↓ (1 = ghost remains) |
| leave-out local-PCA residual | `‖q − proj_local(q)‖ / ‖q − local mean‖`, local PCA on k=64 NN excluding q's own nearest neighbour; reference = real states | fraction | ↓ |
| global-PCA hull residual | `‖h − proj_global(h)‖` onto the global 90%-var subspace; reference = real states | ‖h‖ units | ↓ |

In [ ]:
# [9] §4 — shared edit-set setup (objective-independent): targets, sim renders, ghost/target rays, GT post-edit trajectory.
from pim.simulator.sim import Scene, SimConfig
from pim.simulator.renderer import render_scene

SUBSPACE_VAR, LOCAL_VAR, LOCAL_BANK_SIZE = 0.90, 0.90, 50_000
LOCAL_K_GEO, N_EDIT, N_ROLLOUT, K_GEO_ITERS, N_CTX = 64, 64, 15, 120, 6
N = min(N_EDIT, edits.n_samples); ef = edits.edit_frame

tgt_pos_flat = edits.positions[:N, ef, :N_OBJ, :].reshape(N, N_OBJ * 2).astype(np.float32)
vel_edits = h5py.File(edits.h5_path, "r")["velocities"][:N, ef, :N_OBJ, :].astype(np.float32)
tgt = torch.from_numpy(tgt_pos_flat).float().to(DEVICE)
tgt_pv = torch.from_numpy(np.concatenate([tgt_pos_flat, vel_edits.reshape(N, N_OBJ * 2)], 1)).float().to(DEVICE)

gt_traj_obs = edits.clean_obs[:N, ef:ef + N_ROLLOUT, :].astype(np.float32)   # true post-edit trajectory (sim, never a model output)
ctx_obs = edits.obs[:N, ef - N_CTX:ef, :].astype(np.float32)
gt_obs_edit_frame = torch.from_numpy(edits.clean_obs[:N, ef, :]).float().to(DEVICE)
OBS_RES = gt_traj_obs.shape[-1]

sim = test.config["dataset"]["sim"]
cfg1 = SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"], x_far=sim["x_far"],
                 n_objects=N_OBJ, radius=sim["radius"], n_frames=1, dt=sim["dt"], obs_res=sim["obs_res"],
                 refl_min=sim["refl_min"], refl_max=sim["refl_max"], fixed_reflectivities=True,
                 obs_noise_std=0.0, boundary="open", always_in_frustum=False)
REFL = np.array([sim["refl_min"], sim["refl_max"]], dtype=np.float32)
RAD = np.array([sim["radius"]] * N_OBJ, dtype=np.float32)
COLc = np.tile(np.array([[1, 1, 1]], np.float32), (N_OBJ, 1))
tgt_pos = edits.positions[:N, ef, :N_OBJ, :].astype(np.float32)
pre_pos = edits.positions[:N, ef - 1, :N_OBJ, :].astype(np.float32)
tgt_render_id = np.zeros((N, OBS_RES), np.int64); tgt_render_int = np.zeros((N, OBS_RES), np.float32)
pre_render_id = np.zeros((N, OBS_RES), np.int64)
for i in range(N):
    sc = Scene(positions=tgt_pos[i][None], velocities=np.zeros((1, N_OBJ, 2), np.float32),
               radii=RAD, colors=COLc, reflectivities=REFL, config=cfg1)
    _, rid, rint = render_scene(sc); tgt_render_id[i], tgt_render_int[i] = rid[0], rint[0]
    scp = Scene(positions=pre_pos[i][None], velocities=np.zeros((1, N_OBJ, 2), np.float32),
                radii=RAD, colors=COLc, reflectivities=REFL, config=cfg1)
    _, ridp, _ = render_scene(scp); pre_render_id[i] = ridp[0]

edit_obj = edits.edit_object[:N]
ghost_mask = np.zeros((N, OBS_RES), bool); target_mask = np.zeros((N, OBS_RES), bool)
for i in range(N):
    ghost_mask[i] = (pre_render_id[i] == edit_obj[i]) & (tgt_render_id[i] != edit_obj[i])
    target_mask[i] = (tgt_render_id[i] == edit_obj[i])

def centroid(mask_row):
    idx = np.where(mask_row)[0]; return idx.mean() if idx.size else np.nan
teleport = np.linalg.norm(tgt_pos - pre_pos, axis=-1)[np.arange(N), edit_obj]
has_ghost = ghost_mask.sum(1) >= 3
SAMPLES = list(np.argsort(teleport * has_ghost)[::-1][:3])
print(f"N={N}  edit_frame={ef}  rollout={N_ROLLOUT}  ghost rays available: {int(ghost_mask.sum())}")
print(f"waterfall/scan samples {SAMPLES} (teleport {[round(float(teleport[s]),2) for s in SAMPLES]})")


In [ ]:
# [10] §4 — per-objective prep: warm-up h0 (pre-edit), linear position probe, frozen MLP (pos,vel) probe,
#      global-PCA subspace, local state bank, and the teacher-forced post-edit state (the "True-state swap" reference).
from dataclasses import replace

@torch.no_grad()
def tf_hidden_at(model, obs_seqs, frame):
    """Teacher-force each sequence THROUGH `frame` (inclusive — model sees the teleport frame); return flat states (N,H)."""
    out = np.zeros((obs_seqs.shape[0], H), np.float32)
    for i in range(obs_seqs.shape[0]):
        ot = torch.from_numpy(obs_seqs[i]).float().to(DEVICE); state = None
        for t in range(frame + 1):
            _, state = model.step(ot[t].unsqueeze(0), state)
        out[i] = model.flat_state(state).squeeze(0).cpu().numpy()
    return out

def prep_model(model, states, w):
    sdef = StateDefinition(name="positions", state_shape=(N_OBJ, 2), extract_fn=lambda b: b["positions"])
    lin = LinearExtractor(H, sdef, use_lstsq=True); lin.fit(states, pos_tf, mask=vis_tf, device=DEVICE)
    lin = lin.to(DEVICE).eval(); A, b_, A_pinv = probe_decomposition(lin)
    sdef_pv = StateDefinition(name="posvel", state_shape=(N_OBJ * 4,), extract_fn=lambda b: b)
    mlp_pv = MLPExtractor(H, sdef_pv, mlp_hidden=128, n_epochs=30, lr=5e-3)
    pv_loss = mlp_pv.fit(states, posvel_tf, mask=vis_tf, device=DEVICE); mlp_pv = mlp_pv.to(DEVICE).eval()
    sub = fit_state_subspace(states, var_threshold=SUBSPACE_VAR)
    sub = replace(sub, mean=sub.mean.to(DEVICE), basis=sub.basis.to(DEVICE),
                  explained_variance_ratio=sub.explained_variance_ratio.to(DEVICE))
    bank_all = states.reshape(-1, H)
    bidx = np.random.RandomState(0).choice(bank_all.shape[0], size=min(LOCAL_BANK_SIZE, bank_all.shape[0]), replace=False)
    bank = torch.from_numpy(bank_all[bidx]).float().to(DEVICE)
    warm = eval.warm_up_to_edit(model, edits.obs[:N], ef, n_viz=N, n_ctx_show=8, device=DEVICE)
    h0 = torch.from_numpy(warm.h_at_edit[:N]).float().to(DEVICE)
    h_swap = torch.from_numpy(tf_hidden_at(model, edits.obs[:N], ef)).float().to(DEVICE)
    print(f"RSSM {w}: MLP (pos,vel) probe final train loss {pv_loss:.5f}")
    return dict(model=model, A=A, b=b_, A_pinv=A_pinv, mlp_pv=mlp_pv, sub=sub, bank=bank, h0=h0, h_swap=h_swap)

WM = {w: prep_model(MODELS[w], STATES[w], w) for w in WLABELS}

def readout(P, h): return h @ P["A"].T + P["b"]
def readout_rmse(P, h): return float((readout(P, h) - tgt).pow(2).mean().sqrt())

@torch.no_grad()
def loo_local_resid(h_batch, bank, k_neighbors=LOCAL_K_GEO, leave_out=True, n_probe=100, var_threshold=LOCAL_VAR):
    """||q - proj_local(q)|| / ||q - local mean||, local PCA on k NN EXCLUDING q's own nearest neighbour."""
    hb = h_batch if isinstance(h_batch, torch.Tensor) else torch.as_tensor(h_batch, device=DEVICE, dtype=torch.float32)
    fracs = []
    for i in range(min(n_probe, hb.shape[0])):
        q = hb[i].reshape(-1); d = torch.cdist(q[None], bank)[0]
        kk = k_neighbors + (1 if leave_out else 0)
        idx = torch.topk(d, min(kk, bank.shape[0]), largest=False).indices
        if leave_out: idx = idx[1:]
        sub = _pca_subspace(bank[idx], n_components=None, var_threshold=var_threshold)
        proj = project_to_subspace(q[None], sub)[0]
        fracs.append(float((q - proj).norm()) / max(float((q - sub.mean).norm()), 1e-9))
    return float(np.mean(fracs))

REAL_LOO, REAL_GLOB = {}, {}
for w in WLABELS:
    P = WM[w]
    REAL_LOO[w] = loo_local_resid(P["bank"][:200], P["bank"], n_probe=200)
    REAL_GLOB[w] = float(offmanifold_residual(P["bank"][:2000], P["sub"]).mean())
    print(f"RSSM {w}: un-edited readout RMSE {readout_rmse(P, P['h0']):.4f} | true-state-swap readout RMSE {readout_rmse(P, P['h_swap']):.4f} "
          f"| real-state loo local-PCA resid {REAL_LOO[w]:.3f} | real-state global hull resid {REAL_GLOB[w]:.3f}")


In [ ]:
# [11] §4 — run the five editors on all three RSSMs, then roll each edited/reference state out N_ROLLOUT steps.
from tqdm.auto import tqdm
ED_ORDER = ["Readout injection", "MLP-probe gradient", "Global-PCA projection", "PCA geodesic", "Decoder gradient"]

@torch.no_grad()
def pca_geodesic(P, h_start, target, k_local=LOCAL_K_GEO, const_step=None, k_iters=K_GEO_ITERS, desc="PCA geodesic"):
    A, b_, A_pinv, bank = P["A"], P["b"], P["A_pinv"], P["bank"]
    if const_step is None:
        const_step = 0.34 * float((inject_state(h_start, target, A, A_pinv, b_) - h_start).norm(dim=-1).mean())
    Nn = h_start.shape[0]; h_out = torch.empty_like(h_start); rmse_log = np.full((Nn, k_iters + 1), np.nan)
    for i in tqdm(range(Nn), desc=desc, leave=False):
        h = h_start[i:i + 1]; t = target[i:i + 1]
        rmse_log[i, 0] = float((h @ A.T + b_ - t).pow(2).mean().sqrt())
        for kk in range(k_iters):
            d = inject_state(h, t, A, A_pinv, b_) - h
            nrm = d.norm(); dhat = d / nrm if float(nrm) > 1e-12 else d
            h_step = h + const_step * dhat
            sub = fit_local_subspace(bank, h_step[0], k_neighbors=k_local, var_threshold=LOCAL_VAR, bank_size=LOCAL_BANK_SIZE)
            h = project_to_subspace(h_step, sub)
            rmse_log[i, kk + 1] = float((h @ A.T + b_ - t).pow(2).mean().sqrt())
        h_out[i] = h[0]
    return h_out, rmse_log, const_step

def decoder_grad_edit(model, h_init, target_obs, n_iter=400, lr=0.05):
    h = h_init.clone().detach().requires_grad_(True); opt = torch.optim.Adam([h], lr=lr)
    with torch.backends.cudnn.flags(enabled=False):
        for _ in range(n_iter):
            pred = model.decode(model.state_from_flat(h))
            loss = ((pred - target_obs) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return h.detach(), float(loss.item())

EDITS4, GEO_LOG, CONST_STEP = {}, {}, {}
for w in WLABELS:
    P = WM[w]; h0 = P["h0"]; A, b_, A_pinv = P["A"], P["b"], P["A_pinv"]; E = {}
    E["Readout injection"] = inject_state(h0, tgt, A, A_pinv, b_)
    outs = []
    for i in tqdm(range(N), desc=f"MLP-probe gradient ({w})", leave=False):
        h_i, _ = gradient_steer(h0[i:i + 1], tgt_pv[i:i + 1], P["mlp_pv"], n_steps=200, lr=0.01); outs.append(h_i)
    E["MLP-probe gradient"] = torch.cat(outs, 0)
    E["Global-PCA projection"] = manifold_steer(h0, tgt, lambda h, t: inject_state(h, t, A, A_pinv, b_), P["sub"], n_iters=50)
    E["PCA geodesic"], GEO_LOG[w], CONST_STEP[w] = pca_geodesic(P, h0, tgt, desc=f"PCA geodesic ({w})")
    E["Decoder gradient"], dec_loss = decoder_grad_edit(P["model"], h0, gt_obs_edit_frame)
    EDITS4[w] = E
    print(f"RSSM {w}: const_step={CONST_STEP[w]:.4f} | geodesic readout RMSE {np.nanmean(GEO_LOG[w][:,0]):.3f} -> "
          f"{np.nanmean(GEO_LOG[w][:,-1]):.3f} | decoder-grad decode MSE {dec_loss:.6f}")
    print("   readout RMSE: " + " | ".join(f"{n} {readout_rmse(P, h):.3f}" for n, h in E.items()))

# rollouts from every reference/editor state (rollout step s targets sim frame ef+s)
REF_ORDER = ["Unsteered", "True-state swap"]
COL = {"GT (sim)": "k", "Unsteered": OK["grey"], "True-state swap": "#56B4E9",
       "Readout injection": OK["yellow"], "MLP-probe gradient": OK["orange"],
       "Global-PCA projection": OK["green"], "PCA geodesic": OK["blue"], "Decoder gradient": OK["pink"]}

@torch.no_grad()
def rollout_from_flat(model, h_array, n_rollout):
    out = []
    for i in range(h_array.shape[0]):
        h = torch.as_tensor(h_array[i], dtype=torch.float32, device=DEVICE).unsqueeze(0)
        o, _ = _rollout(model, h, n_rollout); out.append(o)
    return np.stack(out)

ROLL = {}
for w in WLABELS:
    P = WM[w]; states4 = {"Unsteered": P["h0"], "True-state swap": P["h_swap"], **EDITS4[w]}
    ROLL[w] = {n: rollout_from_flat(P["model"], h.detach().cpu().numpy(), N_ROLLOUT) for n, h in states4.items()}
    print(f"RSSM {w}: {len(ROLL[w])} rollout sets, each {ROLL[w]['Unsteered'].shape}")


In [ ]:
# [12] §4 — metric suite (same metrics/units for all three objectives) + per-step tables.
#   readout RMSE (pos), GT next-step RMSE (obs), step-0 vs static target, obs-change (% of true-state swap),
#   ghost-ray ratio, leave-out local-PCA residual, global-PCA hull residual.  References first, then editors.
METRICS, STEP_RMSE, SWAP_CHG, DIST_TO_UNSTEERED = {}, {}, {}, {}
for w in WLABELS:
    P = WM[w]; obs_u = ROLL[w]["Unsteered"]
    swap_chg = rms(ROLL[w]["True-state swap"][:, 0, :], obs_u[:, 0, :]); SWAP_CHG[w] = swap_chg
    rows = {}
    for n in REF_ORDER + ED_ORDER:
        o = ROLL[w][n]
        h = {"Unsteered": P["h0"], "True-state swap": P["h_swap"]}.get(n)
        if h is None: h = EDITS4[w][n]
        chg = rms(o[:, 0, :], obs_u[:, 0, :])
        ghost = float(o[:, 0, :][ghost_mask].mean() / max(obs_u[:, 0, :][ghost_mask].mean(), 1e-6)) if ghost_mask.sum() else float("nan")
        rows[n] = dict(readout=readout_rmse(P, h), nextstep=rms(o[:, 1, :], gt_traj_obs[:, 1, :]),
                       to_tgt0=rms(o[:, 0, :], tgt_render_int), obs_chg=chg,
                       pct_swap=100 * chg / max(swap_chg, 1e-9), ghost=ghost,
                       loo_resid=loo_local_resid(h, P["bank"], n_probe=min(64, N)),
                       glob_resid=float(offmanifold_residual(h, P["sub"]).mean()))
    METRICS[w] = rows
    STEP_RMSE[w] = {n: [rms(ROLL[w][n][:, s, :], gt_traj_obs[:, s, :]) for s in range(N_ROLLOUT)] for n in REF_ORDER + ED_ORDER}
    DIST_TO_UNSTEERED[w] = {n: [rms(ROLL[w][n][:, s, :], obs_u[:, s, :]) for s in range(N_ROLLOUT)] for n in ["True-state swap"] + ED_ORDER}

M4_COLS = [("readout","readout RMSE (pos)","{:.3f}"), ("nextstep","GT next-step RMSE (obs)","{:.3f}"),
           ("to_tgt0","step-0 vs static target (obs)","{:.3f}"), ("obs_chg","obs-change (obs)","{:.3f}"),
           ("pct_swap","% of swap","{:.1f}"), ("ghost","ghost-ray ratio","{:.3f}"),
           ("loo_resid","leave-out local-PCA resid (frac)","{:.3f}"), ("glob_resid","global-PCA hull resid (‖h‖)","{:.3f}")]
for w in WLABELS:
    print(f"=== RSSM {w} — editor metrics (references first) ===")
    print(f"    references: real-state leave-out local-PCA resid {REAL_LOO[w]:.3f} | real-state global hull resid "
          f"{REAL_GLOB[w]:.3f} | true-state-swap obs-change {SWAP_CHG[w]:.4f} (the 100% denominator)")
    display(md_table(METRICS[w], M4_COLS, row_hdr=f"{w} editor"))

# per-step (i) GT-trajectory RMSE and (ii) distance-to-unsteered (reverts vs collapses), all three objectives
sidx = np.arange(N_ROLLOUT)
for w in WLABELS:
    cols_i = [(n, n, "{:.3f}") for n in REF_ORDER + ED_ORDER]
    data_i = {f"step {s}": {n: STEP_RMSE[w][n][s] for n in REF_ORDER + ED_ORDER} for s in sidx}
    cols_ii = [(n, n, "{:.3f}") for n in ["True-state swap"] + ED_ORDER]
    data_ii = {f"step {s}": {n: DIST_TO_UNSTEERED[w][n][s] for n in ["True-state swap"] + ED_ORDER} for s in sidx}
    print(f"=== RSSM {w} — (i) per-step GT-trajectory RMSE: RMSE(gen obs @ step s, sim clean obs @ frame ef+s) ===")
    display(md_table(data_i, cols_i, row_hdr="rollout step"))
    print(f"=== RSSM {w} — (ii) distance to unsteered rollout (small+small(i)=reverts; large=collapses off-distribution) ===")
    display(md_table(data_ii, cols_ii, row_hdr="rollout step"))


In [ ]:
# [13] Fig 4 — §4 editing head-to-head, one ROW per objective w:
#   (left) readout accuracy vs GT next-step observation accuracy; (mid) per-step GT-trajectory RMSE;
#   (right) manifold residency (leave-out local-PCA residual) of the edited state vs the real-state reference.
from matplotlib.lines import Line2D
steps = np.arange(N_ROLLOUT)
nrow = len(WLABELS)
fig, axes = plt.subplots(nrow, 3, figsize=(18, 4.4 * nrow))
for r, w in enumerate(WLABELS):
    Mx = METRICS[w]; pa, pb, pc = axes[r]
    for n in REF_ORDER + ED_ORDER:
        pa.scatter(Mx[n]["readout"], Mx[n]["nextstep"], s=95, color=COL[n],
                   marker="s" if n in REF_ORDER else "o", edgecolor="k", zorder=3)
    pa.set_xlabel("readout RMSE at edit step (position units)")
    pa.set_ylabel("GT next-step RMSE (obs intensity)")
    pa.set_title(f"({'adg'[r]}) RSSM {w} — readout vs next-step obs accuracy"); style_ax(pa)
    for n in REF_ORDER + ED_ORDER:
        pb.plot(steps, STEP_RMSE[w][n], color=COL[n], lw=1.6, marker="o", ms=3)
    pb.set_xlabel("rollout step (0 = edit frame)")
    pb.set_ylabel("RMSE(gen obs, true post-edit obs)")
    pb.set_title(f"({'beh'[r]}) RSSM {w} — per-step GT-trajectory RMSE"); style_ax(pb)
    names_c = REF_ORDER + ED_ORDER
    vals = [Mx[n]["loo_resid"] for n in names_c]
    pc.bar(range(len(names_c)), vals, color=[COL[n] for n in names_c], alpha=0.9)
    for i, v in enumerate(vals): pc.text(i, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)
    pc.axhline(REAL_LOO[w], color="0.3", ls="--", lw=1.4)
    pc.text(len(names_c) - 0.55, REAL_LOO[w] + 0.015, f"real states {REAL_LOO[w]:.2f}", ha="right", fontsize=8, color="0.3")
    pc.set_xticks(range(len(names_c))); pc.set_xticklabels(names_c, rotation=25, ha="right", fontsize=8)
    pc.set_ylabel("leave-out local-PCA residual (fraction)")
    pc.set_title(f"({'cfi'[r]}) RSSM {w} — manifold residency of the edited state"); style_ax(pc)
handles = [Line2D([0],[0], marker="s" if n in REF_ORDER else "o", linestyle="none", markersize=8,
                  markerfacecolor=COL[n], markeredgecolor="k", label=n) for n in REF_ORDER + ED_ORDER]
fig.legend(handles=handles, loc="upper center", ncol=7, fontsize=9, frameon=False, bbox_to_anchor=(0.5, 0.995))
fig.suptitle("Fig 4 — Editing head-to-head across training objectives (rows: w=1 single-step, w=2, w=5 multi-step)",
             y=1.0, fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.955])
fig.savefig(f"{OUT}/fig4_editor_metrics.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)


In [ ]:
# [14] Fig 5 — §4 editor waterfalls (dark theme) + Fig 6 — step-0 observation scans, one block per objective w.
#   Waterfall: 6 sim clean-obs context rows, then the model rollout below the edit-frame line. GT (sim) column = the
#   simulation itself (never a model output). green = target (post-edit) location; red dashed = ghost (pre-edit) location.
from matplotlib.lines import Line2D
DARK_BG, DARK_TEXT, DARK_TICK, EDIT_LINE = "#0a0a14", "#a3adc2", "#808a9d", "#fa8850"
WATERFALL_COLS = ["GT (sim)"] + REF_ORDER + ED_ORDER

def editor_waterfall_fig(w, fig_tag, fname):
    n_cols = len(WATERFALL_COLS)
    fig, axes = plt.subplots(len(SAMPLES), n_cols, figsize=(3.0 * n_cols, 3.4 * len(SAMPLES)),
                             squeeze=False, facecolor=DARK_BG)
    for r, smp in enumerate(SAMPLES):
        tgt_cx = centroid(tgt_render_id[smp] == edit_obj[smp])
        pre_cx = centroid(pre_render_id[smp] == edit_obj[smp])
        for c, n in enumerate(WATERFALL_COLS):
            ax = axes[r][c]
            post = gt_traj_obs[smp] if n == "GT (sim)" else ROLL[w][n][smp]
            panel = np.clip(np.concatenate([ctx_obs[smp], post], axis=0), 0, 1)
            ax.set_facecolor(DARK_BG)
            for sp in ax.spines.values():
                sp.set_edgecolor(DARK_TICK)
            ax.imshow(panel, aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1, interpolation="nearest")
            ax.axhline(N_CTX - 0.5, color=EDIT_LINE, lw=1.2, ls="--", alpha=0.8)
            if not np.isnan(tgt_cx): ax.axvline(tgt_cx, color="#00E676", lw=1.6, alpha=0.9)
            if not np.isnan(pre_cx): ax.axvline(pre_cx, color="#FF5252", ls="--", lw=1.6, alpha=0.9)
            if r == 0: ax.set_title(n, fontsize=9, color=DARK_TEXT)
            if c == 0:
                ax.set_ylabel(f"sample {smp} (teleport {teleport[smp]:.2f})\nsim frame", fontsize=8, color=DARK_TEXT)
                ax.set_yticks([0, N_CTX, N_CTX + 5, N_CTX + 10]); ax.set_yticklabels([ef - N_CTX, ef, ef + 5, ef + 10])
            else:
                ax.set_yticks([])
            ax.set_xlabel("ray", fontsize=8, color=DARK_TEXT); ax.tick_params(colors=DARK_TICK, labelsize=7)
    handles = [Line2D([0],[0], color="#00E676", lw=2.2, label="target (post-edit)"),
               Line2D([0],[0], color="#FF5252", ls="--", lw=2.2, label="ghost (pre-edit)"),
               Line2D([0],[0], color=EDIT_LINE, ls="--", lw=2.2, label=f"edit frame ({N_CTX} context rows above)")]
    fig.legend(handles=handles, loc="upper center", ncol=3, fontsize=10, frameon=False,
               labelcolor=DARK_TEXT, bbox_to_anchor=(0.5, 0.975))
    fig.suptitle(f"Fig {fig_tag} — RSSM {w} editor waterfalls: {N_CTX} sim context frames, then model rollout from the edited state",
                 y=0.998, fontsize=12, color=DARK_TEXT)
    fig.tight_layout(rect=[0, 0, 1, 0.945])
    fig.savefig(f"{OUT}/{fname}", dpi=125, bbox_inches="tight", facecolor=DARK_BG); display(fig); plt.close(fig)

def scan_fig(w, fig_tag, fname):
    rays = np.arange(OBS_RES)
    fig, axes = plt.subplots(len(SAMPLES), 1, figsize=(11.5, 3.0 * len(SAMPLES)), squeeze=False)
    for r, smp in enumerate(SAMPLES):
        ax = axes[r][0]
        ax.plot(rays, tgt_render_int[smp], color="k", ls="--", lw=1.6, label="target render (static, edit frame)", zorder=6)
        gz = np.where(ghost_mask[smp])[0]
        if gz.size: ax.axvspan(gz.min()-0.5, gz.max()+0.5, color="red", alpha=0.10, zorder=0, label="ghost zone")
        tz = np.where(target_mask[smp])[0]
        if tz.size: ax.axvspan(tz.min()-0.5, tz.max()+0.5, color="green", alpha=0.10, zorder=0, label="target zone")
        for n in REF_ORDER + ED_ORDER:
            ax.plot(rays, ROLL[w][n][smp, 0], color=COL[n], lw=1.5, alpha=0.9, label=n, zorder=3)
        ax.set_title(f"sample {smp} (edited object {edit_obj[smp]}, teleport {teleport[smp]:.2f})", fontsize=10)
        ax.set_xlabel("ray index"); ax.set_ylabel("intensity"); ax.set_ylim(-0.02, 1.05); style_ax(ax)
        if r == 0: ax.legend(fontsize=7, ncol=3, loc="upper right")
    fig.suptitle(f"Fig {fig_tag} — RSSM {w}: generated observation at rollout step 0 (direct edit), per editor", y=1.0, fontsize=12)
    fig.tight_layout(); fig.savefig(f"{OUT}/{fname}", dpi=125, bbox_inches="tight"); display(fig); plt.close(fig)

for k, w in enumerate(WLABELS):
    tag = k + 1
    editor_waterfall_fig(w, f"5{'abc'[k]}", f"fig5{'abc'[k]}_waterfalls_{w.replace('=','')}.png")
    scan_fig(w, f"6{'abc'[k]}", f"fig6{'abc'[k]}_scans_{w.replace('=','')}.png")


---
## §5 — Summary: what the multi-step objective changes

Consolidated tables (below) and the headline read. **Interpretation is confined here.** The organizing question: relative to the single-step baseline (`w=1`), does adding a free-running rollout term to the training objective move any of §0 sharpness, §1 geometry, §2 recoverability, §3 canonicality, or — the headline — **§4 editability**? And does it cost predictive sharpness (blur / mode-collapse)?

In [ ]:
# [15] §5 — consolidated summary across w (every headline number) + PNG manifest.
from IPython.display import Markdown, display

display(Markdown("## Multi-step objective — consolidated summary (RSSM w=1 / w=2 / w=5)\n"
                 "Every number recomputed in this run; formulas in the definitions table. RMSE throughout."))

# --- §0 sharpness / next-step quality ---
display(Markdown("**§0 Sharpness & next-step predictive quality** — does the multi-step objective blur the decoder?"))
S_COLS = [("next_rmse_clean","next-step RMSE vs clean (obs)","{:.4f}"),
          ("horizon_rmse","open-loop horizon RMSE (mean, obs)","{:.4f}"),
          ("tv_ratio","rollout sharpness (TV / GT TV)","{:.3f}")]
srows = {w: dict(next_rmse_clean=NEXT[w]["rmse_clean"], horizon_rmse=float(np.mean(HRMSE[w])),
                 tv_ratio=TV_ROLL[w]/GT_TV_ROLL) for w in WLABELS}
display(md_table(srows, S_COLS, row_hdr="objective"))

# --- §1 geometry ---
display(Markdown("**§1 Geometry** — dimensionality & curvature of the visited-state manifold (physical DOF = 8)."))
GEO_COLS = [("hull90","PCA hull dim @90%","{:d}"),("twonn","intrinsic dim (TwoNN)","{:.1f}"),
            ("mle","intrinsic dim (MLE)","{:.1f}"),("tangent","tangent rotation @NN (deg)","{:.0f}")]
grows = {w: dict(hull90=GEO[w]["dims"][0.90], twonn=GEO[w]["twonn"], mle=GEO[w]["mle"], tangent=GEO[w]["tangent"]) for w in WLABELS}
display(md_table(grows, GEO_COLS, row_hdr="objective"))

# --- §2 recoverability ---
display(Markdown("**§2 Recoverability** — probe R² for reading (pos,vel) from a single hidden state (late-t; in-sample)."))
REC_COLS = [("pos_lin","position R² (linear)","{:.2f}"),("pos_mlp","position R² (MLP)","{:.2f}"),
            ("vel_lin","velocity R² (single-frame linear)","{:.2f}"),("vel_mlp","velocity R² (single-frame MLP)","{:.2f}"),
            ("vel_gain","two-frame − single-frame (MLP)","{:+.3f}")]
rrows = {}
for w in WLABELS:
    o = VEL[(w,"late")]
    rrows[w] = dict(pos_lin=POS[(w,"lin")]["r2"], pos_mlp=POS[(w,"mlp")]["r2"],
                    vel_lin=o[("sf","lin")]["r2"], vel_mlp=o[("sf","mlp")]["r2"],
                    vel_gain=o[("win","mlp")]["r2"]-o[("sf","mlp")]["r2"])
display(md_table(rrows, REC_COLS, row_hdr="objective"))

# --- §3 canonicality ---
display(Markdown("**§3 Canonicality** — fiber residual ‖h − g(pos,vel)‖ / ‖h‖ (0 = h fully a function of the 8-dim statistic)."))
FIB_COLS = [("lin","linear g: residual frac","{:.3f}"),("mlp","MLP g: residual frac","{:.3f}"),("r2","MLP g: R² on h","{:.3f}")]
frows = {w: dict(lin=FIBER[w]["linear"][0], mlp=FIBER[w]["mlp"][0], r2=FIBER[w]["mlp"][1]) for w in WLABELS}
display(md_table(frows, FIB_COLS, row_hdr="objective"))

# --- §4 editing head-to-head ---
display(Markdown("**§4 Editing head-to-head** — same edit set on all three RSSMs; the true-state swap is the upper bound any hidden-state editor could reach."))
SUM4_COLS = [("readout","readout RMSE (pos)","{:.3f}"),("nextstep","GT next-step RMSE (obs)","{:.3f}"),
             ("pct_swap","obs-change (% of swap)","{:.1f}"),("ghost","ghost-ray ratio","{:.3f}"),
             ("loo_resid","leave-out local-PCA resid (frac)","{:.2f}")]
for w in WLABELS:
    display(Markdown(f"_{w}: true-state-swap obs-change {SWAP_CHG[w]:.3f} (the 100%); real-state leave-out local-PCA resid "
                     f"{REAL_LOO[w]:.2f}; unsteered GT next-step RMSE {METRICS[w]['Unsteered']['nextstep']:.3f}._"))
    display(md_table({n: METRICS[w][n] for n in REF_ORDER+ED_ORDER}, SUM4_COLS, row_hdr=f"{w} — state"))

# --- headline read + PNG manifest ---
head = []
head.append(f"next-step RMSE vs clean: " + ", ".join(f"{w} {NEXT[w]['rmse_clean']:.4f}" for w in WLABELS))
head.append(f"rollout sharpness (TV/GT): " + ", ".join(f"{w} {TV_ROLL[w]/GT_TV_ROLL:.2f}" for w in WLABELS))
head.append(f"intrinsic dim (TwoNN): " + ", ".join(f"{w} {GEO[w]['twonn']:.1f}" for w in WLABELS))
head.append(f"fiber residual (MLP): " + ", ".join(f"{w} {FIBER[w]['mlp'][0]:.3f}" for w in WLABELS))
head.append(f"best non-oracle editor GT next-step RMSE / unsteered: " +
            ", ".join(f"{w} {min(METRICS[w][n]['nextstep'] for n in ED_ORDER if n!='Decoder gradient'):.3f}/{METRICS[w]['Unsteered']['nextstep']:.3f}" for w in WLABELS))
print("HEADLINE NUMBERS")
for h in head: print("  " + h)

png_paths = sorted(os.path.join(OUT, f) for f in os.listdir(OUT) if f.endswith(".png"))
display(Markdown("**Figure PNG manifest** (regenerated by this run):\n\n" + "\n".join(f"- `{p}`" for p in png_paths)))
